In [1]:
from IPython.display import HTML
HTML('''
    <style> body {font-family: "Roboto Condensed Light", "Roboto Condensed";} h2 {padding: 10px 12px; background-color: #E64626; position: static; color: #ffffff; font-size: 40px;} .text_cell_render p { font-size: 15px; } .text_cell_render h1 { font-size: 30px; } h1 {padding: 10px 12px; background-color: #E64626; color: #ffffff; font-size: 40px;} .text_cell_render h3 { padding: 10px 12px; background-color: #0148A4; position: static; color: #ffffff; font-size: 20px;} h4:before{ 
    content: "@"; font-family:"Wingdings"; font-style:regular; margin-right: 4px;} .text_cell_render h4 {padding: 8px; font-family: "Roboto Condensed Light"; position: static; font-style: italic; background-color: #FFB800; color: #ffffff; font-size: 18px; text-align: center; border-radius: 5px;}input[type=submit] {background-color: #E64626; border: solid; border-color: #734036; color: white; padding: 8px 16px; text-decoration: none; margin: 4px 2px; cursor: pointer; border-radius: 20px;}</style>
''')

# School Locations and Ranking

First step is to load in all the required Packages

In [2]:
import requests
from bs4 import BeautifulSoup
import html5lib
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement
import matplotlib.pyplot as plt

## Location Dataset

Next we need the list of schools (names) with locations attached. This includes all 3300 schools in NSW (Make sure the related CSV is in the same directory as this notebook). This is read in, its two coordinates columns converted into a single geometry column, and finally stripped back to just contain the necessary data.

In [3]:
Initial_data_path = 'NSW_Schools.csv'
school_location_data = pd.read_csv(Initial_data_path)
latitudes = school_location_data['Latitude']
longitudes = school_location_data['Longitude']
school_location_data = gpd.GeoDataFrame(school_location_data, geometry=gpd.points_from_xy(longitudes, latitudes)) #Pairs together Longitude Latitude pairs and turns them into geometry points
school_location_data = school_location_data[['School Name','geometry']] # Keep only important columns
school_location_data.head()

,School Name,geometry
0,A.G.B.U. Alexander Primary School,POINT (151.18715 -33.67679)
1,Abbotsford Public School,POINT (151.13100 -33.85270)
2,Abbotsleigh,POINT (151.11100 -33.71990)
3,Abbotsleigh Junior School,POINT (151.11300 -33.71620)
4,Aberdeen Public School,POINT (150.88800 -32.16610)


## Scraping the Ranking Data (School Names)

Next we begin scraping the data containing the HSC 'results' of numerous schools in NSW.

In [4]:
basic_URL = "https://bettereducation.com.au/Results/Hsc.aspx" #This is where the data is coming from
try:
    webpage_source = requests.get(basic_URL).text
except:
    print('Error Occured: Request Failed')
try:
    content = BeautifulSoup(webpage_source, 'html5lib')
except:
        print('Error Occured: Parsing Failed')

This webpage contains a table with ~200 schools, where the data is contained within tags of a similar id. To efficiently extract these I created a function that takes in a number (Row Number) and returns the related id for that school's name.

In [5]:
def id_constructor(start='ctl00_ContentPlaceHolder1_GridView1_ctl',end='_HyperLinkSchool',number='0'):
    new_id = str(start) + str(number) + str(end)
    return new_id

Next we use this function to get the id's for all the rows of the table. This is where we go overboard, and create id's for up to 300 schools, as the exact number of rows in the table is somewhat confusing actually know. On the website you can view 299 schools, but the table says it only has 205.

In [6]:
ids = [] #This will be used each time we iterate over the table.
for i in range(300):
    ids.append(str(str(i).zfill(2))) # zfill is used to pad with 0s required by the tag.

name_ids = []
for i in ids:
    name_ids.append(id_constructor(number=i))

Now we come to actually extracting the names, and storing them in a list that will later be turned into a database column.

In [7]:
school_names_rankings = []
for code in name_ids:
    try:    # try loop is used in case we have got too many ids, and we try to access a row that doesn't exist.
        name = content.find('a', id=code).text
        school_names_rankings.append(name)
    except:
        continue

By checking the length I can check that I have extracted 205 school names, which is partially what is expected from this table. So long as this remains consistent, everything is fine.

In [8]:
len(school_names_rankings)

205

## Scraping the Ranking Data (%DAs)

Next step is to repeat the above process, only this time extracting the data on 'Percentage of exams sat that achieved a DA'. This will be our metric of school success. This begins by creating all the ids for each row of this column of data.

In [9]:
percentDA_ids = []
for i in ids:
    percentDA_ids.append(id_constructor(start='ctl00_ContentPlaceHolder1_GridView1_ctl', end='_lblPercent', number=i))

As before I have a list to store all the results that will later be turned into a column. Iterate over the each of the ids from before to get each entry.

In [10]:
school_percentageDA = []
for code in percentDA_ids:
    try:    # Once again using a try to account for the excess of ids.
        percent_DA = content.find('span', id=code).text
        school_percentageDA.append(percent_DA)
    except:
        continue

## Scraping the HSC Students Data

In addition to the %DA, the website contains information on the number of enrolled HSC students of each school mentioned. This is valuable in determining how 'Bustling' a school is, as more students strongly implies the school is more bustling.

In [11]:
hscStudents_ids = []
for i in ids:
    hscStudents_ids.append(id_constructor(end='_lblStudents', number=i))

Store all the Student Numbers in a list that will be added as a column, iterating over each of the ids made above.

In [12]:
school_HSCstudents = []
for code in hscStudents_ids:
    try:    # Once again using a try to account for the excess of ids.
        HSCstudents = content.find('span', id=code).text
        school_HSCstudents.append(HSCstudents)
    except:
        continue

## Scraping the number of Exams Sat

While the number of students is a good indicator of bustling, 'bustling' is better defined as full of activity, and what activity better than sitting tests. So this will be another good indicator of how 'bustling' the schools actually are.

In [13]:
testSat_ids = []
for i in ids:
    testSat_ids.append(id_constructor(end='_lblExamsSat', number=i))

Once again using the ids to extract each table entry.

In [14]:
school_testSat = []
for code in testSat_ids:
    try:    # Once again using a try to account for the excess of ids.
        testSat = content.find('span', id=code).text
        school_testSat.append(testSat)
    except:
        continue

## Matching Names

Testing for names that are duplicates by turning the list into a set (which removes duplicates). As the lengths are the same, we can assume there are no duplicates.

In [21]:
len(school_names_rankings), len(set(school_names_rankings))

(205, 205)

Attempting to see how many of the school names line up across datasets. Without any alterations, there are 154 that match, and 51 schools that don't match up. These will be the ones we have to identify manually.

In [22]:
matched_schools = list(set(school_names_rankings) & set(school_location_data['School Name']))
unmatched_schools = list(set(school_names_rankings) - set(school_location_data['School Name']))
len(matched_schools), len(unmatched_schools)

(154, 51)

In [23]:
unmatched_schools_df = pd.DataFrame(unmatched_schools)
unmatched_schools_df.to_csv('Schools_to_be_renamed.csv')

A new file containing the incorrect names paired with the correct names was painfully put together, so as to ensure correct mapping, as any level of automation ended up resulting in some incorrections.

In [24]:
unmatched_schools_correct_names = pd.read_csv('Schools_with_new_names.csv')
unmatched_schools_correct_names = unmatched_schools_correct_names.drop(['Unnamed: 0'], axis=1) # Remove the extra 
incorrect_names = list(unmatched_schools_correct_names['Original Name'])
correct_names = list(unmatched_schools_correct_names['New Name'])

Go through each incorrect name in the original list and replace it with the correct one.

In [25]:
for i in range(len(incorrect_names)):
    index = school_names_rankings.index(incorrect_names[i])
    school_names_rankings[index] = correct_names[i]

We expect to have 203 schools as 2 schools could not be matched (Actual location ACT, not NSW).

In [26]:
len(set(school_names_rankings) & set(school_location_data['School Name']))

203

## Storing Data

Storing the data in a dataframe, which will soon be attached to the location GeoDataFrame.

In [27]:
school_ranking_data = {
    'school_name': school_names_rankings,
    'HSC_students': school_HSCstudents,
    'exams_sat': school_testSat,
    'school_achievement': school_percentageDA
}
school_ranking_data = pd.DataFrame(school_ranking_data)
school_ranking_data.head()

,school_name,HSC_students,exams_sat,school_achievement
0,North Sydney Boys High School,158,955,69.0%
1,James Ruse Agricultural High School,158,996,60.0%
2,Baulkham Hills High School,200,1210,55.8%
3,North Sydney Girls High School,153,909,54.7%
4,Reddam House,122,684,54.2%


## Combining Data

The next step is to actually combine the two datasets, we've been thorough in the previous step to ensure that they can be properly joined by the name columns.

In [28]:
school_complete_data = school_ranking_data.merge(
    right = school_location_data, 
    how = 'inner',
    left_on = 'school_name',
    right_on = 'School Name')
school_complete_data = school_complete_data.drop(['School Name'], axis=1) # Remove the duplicate column
school_complete_data = gpd.GeoDataFrame(school_complete_data) # Make the dataframe a Geodataframe
school_complete_data.set_geometry('geometry',inplace=True) # Make sure the correct column is the geometry column
school_complete_data.head()

,school_name,HSC_students,exams_sat,school_achievement,geometry
0,North Sydney Boys High School,158,955,69.0%,POINT (151.20700 -33.82960)
1,James Ruse Agricultural High School,158,996,60.0%,POINT (151.04300 -33.78060)
2,Baulkham Hills High School,200,1210,55.8%,POINT (150.99100 -33.75020)
3,North Sydney Girls High School,153,909,54.7%,POINT (151.20300 -33.83020)
4,Reddam House,122,684,54.2%,POINT (151.25300 -33.88930)


Finally with our complete data we can save it somewhere.

In [29]:
school_complete_data.to_file('School_complete_data.shp')

C:\Users\Scott\AppData\Local\Temp\ipykernel_6940\1294767551.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  school_complete_data.to_file('School_complete_data.shp')


To test whether it was correctly saved, we can load in the file and check the details.

In [36]:
school_complete_data2 = gpd.read_file('School_complete_data.shp')
school_complete_data2

,school_nam,HSC_studen,exams_sat,school_ach,geometry
0,North Sydney Boys High School,158,955,69.0%,POINT (151.20700 -33.82960)
1,James Ruse Agricultural High School,158,996,60.0%,POINT (151.04300 -33.78060)
2,Baulkham Hills High School,200,1210,55.8%,POINT (150.99100 -33.75020)
3,North Sydney Girls High School,153,909,54.7%,POINT (151.20300 -33.83020)
4,Reddam House,122,684,54.2%,POINT (151.25300 -33.88930)
...,...,...,...,...,...
200,Broughton Anglican College,96,531,5.8%,POINT (150.76800 -34.09970)
201,St Columba's Catholic College,120,659,5.8%,POINT (150.58800 -33.67320)
202,Cedars Christian College,45,228,5.7%,POINT (150.82900 -34.45550)
203,St Philip's Christian College - Port Stephens,44,232,5.6%,POINT (152.11000 -32.73830)
